# Data Exploration and Cleaning

This notebook performs initial data exploration and cleaning of the Seattle Airbnb dataset.

**Objectives:**
- Load and inspect the raw data
- Handle missing values and outliers
- Perform initial data quality checks
- Prepare clean dataset for modeling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

%matplotlib inline

## 1. Load Data

In [ ]:
# Load listings data
listings = pd.read_csv('../data/raw/listings.csv')
print(f"Loaded {len(listings):,} listings")
print(f"Columns: {listings.shape[1]}")

# Display first few rows
listings.head()

## 2. Data Quality Assessment

In [ ]:
# Check data types and missing values
print("\n=== Data Info ===")
listings.info()

print("\n=== Missing Values ===")
missing = listings.isnull().sum()
missing_pct = 100 * missing / len(listings)
missing_df = pd.DataFrame({
    'Missing Count': missing[missing > 0],
    'Percentage': missing_pct[missing > 0]
}).sort_values('Percentage', ascending=False)
print(missing_df)

## 3. Price Analysis

In [ ]:
# Extract numeric price if needed
if listings['price'].dtype == 'object':
    listings['price_clean'] = listings['price'].str.replace('$', '').str.replace(',', '').astype(float)
else:
    listings['price_clean'] = listings['price']

# Summary statistics
print("\n=== Price Statistics ===")
print(listings['price_clean'].describe())

# Visualize price distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(listings['price_clean'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Price per Night ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Price Distribution (Raw)')
axes[0].axvline(listings['price_clean'].median(), color='red', linestyle='--', label=f'Median: ${listings["price_clean"].median():.2f}')
axes[0].legend()

# Log scale histogram
log_prices = np.log(listings['price_clean'].dropna())
axes[1].hist(log_prices, bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('Log(Price)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Price Distribution (Log Scale)')

plt.tight_layout()
plt.show()

## 4. Outlier Detection and Handling

In [ ]:
# Identify extreme outliers
Q1 = listings['price_clean'].quantile(0.25)
Q3 = listings['price_clean'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 3 * IQR
upper_bound = Q3 + 3 * IQR

outliers = listings[(listings['price_clean'] < lower_bound) | (listings['price_clean'] > upper_bound)]
print(f"\nFound {len(outliers)} extreme outliers")
print(f"Price range: ${lower_bound:.2f} - ${upper_bound:.2f}")

# Filter reasonable price range
listings_clean = listings[(listings['price_clean'] >= 10) & (listings['price_clean'] <= 1000)].copy()
print(f"\nAfter filtering: {len(listings_clean):,} listings ({100*len(listings_clean)/len(listings):.1f}%)")

## 5. Neighborhood Analysis

In [ ]:
# Analyze neighborhoods
print("\n=== Neighborhood Summary ===")
neighborhood_col = 'neighbourhood_cleansed' if 'neighbourhood_cleansed' in listings_clean.columns else 'neighbourhood'

neighborhood_stats = listings_clean.groupby(neighborhood_col).agg({
    'price_clean': ['count', 'mean', 'median', 'std'],
    'id': 'count'
}).round(2)

neighborhood_stats.columns = ['Count', 'Mean Price', 'Median Price', 'Std Dev', 'Total Listings']
neighborhood_stats = neighborhood_stats.sort_values('Mean Price', ascending=False)
print(neighborhood_stats.head(10))

# Visualize top neighborhoods
top_neighborhoods = neighborhood_stats.head(15)
fig, ax = plt.subplots(figsize=(12, 6))
top_neighborhoods['Mean Price'].plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Average Price per Night ($)')
ax.set_title('Top 15 Neighborhoods by Average Price')
plt.tight_layout()
plt.show()

## 6. Feature Selection for Modeling

In [ ]:
# Select key features for modeling
key_features = [
    'id', 'price_clean', neighborhood_col, 'accommodates', 
    'bedrooms', 'beds', 'bathrooms', 'property_type',
    'room_type', 'number_of_reviews', 'review_scores_rating'
]

# Filter to available columns
available_features = [f for f in key_features if f in listings_clean.columns]
modeling_data = listings_clean[available_features].copy()

print(f"\n=== Modeling Dataset ===")
print(f"Shape: {modeling_data.shape}")
print(f"\nFeatures selected: {', '.join(available_features)}")

# Check completeness
print(f"\nMissing values:")
print(modeling_data.isnull().sum())

## 7. Save Cleaned Data

In [ ]:
# Create processed data directory if needed
import os
os.makedirs('../data/processed', exist_ok=True)

# Save cleaned dataset
modeling_data.to_csv('../data/processed/listings_clean.csv', index=False)
print("\nCleaned data saved to: data/processed/listings_clean.csv")

## Summary

**Key Findings:**
- Loaded and cleaned Seattle Airbnb dataset
- Filtered extreme outliers (price range: $10-$1000)
- Identified neighborhood-level patterns
- Prepared dataset for Bayesian modeling

**Next Steps:**
- Exploratory data analysis (EDA)
- Feature engineering
- Hierarchical Bayesian model development